In [11]:
#Load and validate the modeling inputs

from pathlib import Path
import json

import numpy as np
import pandas as pd
import sklearn

# Reproducibility
RANDOM_STATE = 42

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

# Notebook is located inside the notebooks folder
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "diabetic_modeling_data_final.csv"
)

SCHEMA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "model_feature_schema.json"
)

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT.resolve())
print("Dataset exists:", DATA_PATH.exists())
print("Feature schema exists:", SCHEMA_PATH.exists())

# Stop if required files are missing
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH.resolve()}"
    )

if not SCHEMA_PATH.exists():
    raise FileNotFoundError(
        f"Feature schema not found at: {SCHEMA_PATH.resolve()}"
    )

# Load the audited dataset
df = pd.read_csv(DATA_PATH)

# Load the feature schema created in Notebook 2
with open(
    SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as schema_file:
    feature_schema = json.load(schema_file)

# Extract schema information
TARGET = feature_schema["target_column"]
GROUP_COLUMN = feature_schema["group_split_column"]

identifier_columns = feature_schema["identifier_columns"]
numeric_model_features = feature_schema["numeric_model_features"]
categorical_model_features = feature_schema[
    "categorical_model_features"
]
model_features = feature_schema["all_model_features"]

# Create modeling and tracking objects
X = df[model_features].copy()
y = df[TARGET].copy()
groups = df[GROUP_COLUMN].copy()
tracking_ids = df["encounter_id"].copy()

# Dataset validation
assert df.shape == (99343, 46), (
    f"Unexpected dataset shape: {df.shape}"
)

assert df.isna().sum().sum() == 0, (
    "Missing values were found in the audited dataset."
)

assert df.duplicated().sum() == 0, (
    "Duplicate rows were found in the audited dataset."
)

# Feature validation
assert X.shape == (99343, 43), (
    f"Unexpected predictor shape: {X.shape}"
)

assert len(numeric_model_features) == 8, (
    f"Expected 8 numeric features, found "
    f"{len(numeric_model_features)}."
)

assert len(categorical_model_features) == 35, (
    f"Expected 35 categorical features, found "
    f"{len(categorical_model_features)}."
)

assert len(model_features) == 43, (
    f"Expected 43 total predictors, found "
    f"{len(model_features)}."
)

# Leakage checks
assert TARGET not in X.columns, (
    "Target variable was included in predictors."
)

assert "patient_nbr" not in X.columns, (
    "Patient identifier was included in predictors."
)

assert "encounter_id" not in X.columns, (
    "Encounter identifier was included in predictors."
)

assert set(y.unique()) == {0, 1}, (
    f"Unexpected target values: {sorted(y.unique())}"
)

assert X.shape[0] == y.shape[0] == groups.shape[0], (
    "X, y, and groups contain different row counts."
)

print("\nModeling inputs loaded successfully.")
print("-" * 60)

print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Random state:", RANDOM_STATE)

print("\nDataset shape:", df.shape)
print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Patient-group shape:", groups.shape)
print("Tracking-ID shape:", tracking_ids.shape)

print("\nUnique patients:", groups.nunique())
print("Numeric predictors:", len(numeric_model_features))
print("Categorical predictors:", len(categorical_model_features))
print("Total predictors:", len(model_features))

print("\nTarget distribution:")
print(
    y.value_counts()
    .sort_index()
    .rename(index={
        0: "Not readmitted",
        1: "Readmitted"
    })
)

print(
    f"\nPositive-class rate: {y.mean():.4%}"
)

print(
    "\ncompleted successfully."
)

#Create patient-level train, validation, and test splits

from sklearn.model_selection import train_test_split

# Create one row per patient for group-aware splitting

patient_split_table = (
    df.groupby(GROUP_COLUMN)
    .agg(
        patient_has_readmission=(TARGET, "max"),
        encounter_count=("encounter_id", "size")
    )
    .reset_index()
)

# Group patients by encounter frequency
patient_split_table["encounter_frequency_group"] = np.select(
    [
        patient_split_table["encounter_count"] == 1,
        patient_split_table["encounter_count"] == 2
    ],
    [
        "1_encounter",
        "2_encounters"
    ],
    default="3_or_more_encounters"
)

# Create a patient-level stratification label using:
# 1. Whether the patient ever had a 30-day readmission
# 2. How many encounters the patient has
patient_split_table["split_stratum"] = (
    patient_split_table["patient_has_readmission"].astype(str)
    + "_"
    + patient_split_table["encounter_frequency_group"]
)

print("Patient-level split table created.")
print("Unique patients:", len(patient_split_table))

print("\nPatient split strata:")
print(
    patient_split_table["split_stratum"]
    .value_counts()
    .sort_index()
)

# First split: 70% train, 30% temporary

train_patient_table, temporary_patient_table = train_test_split(
    patient_split_table,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=patient_split_table["split_stratum"]
)


# Second split: 15% validation, 15% test

validation_patient_table, test_patient_table = train_test_split(
    temporary_patient_table,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temporary_patient_table["split_stratum"]
)


# Convert patient IDs to sets
train_patient_ids = set(
    train_patient_table[GROUP_COLUMN]
)

validation_patient_ids = set(
    validation_patient_table[GROUP_COLUMN]
)

test_patient_ids = set(
    test_patient_table[GROUP_COLUMN]
)

# Assign encounter-level rows using patient IDs

train_mask = groups.isin(train_patient_ids)
validation_mask = groups.isin(validation_patient_ids)
test_mask = groups.isin(test_patient_ids)

X_train = X.loc[train_mask].copy()
X_validation = X.loc[validation_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_validation = y.loc[validation_mask].copy()
y_test = y.loc[test_mask].copy()

groups_train = groups.loc[train_mask].copy()
groups_validation = groups.loc[validation_mask].copy()
groups_test = groups.loc[test_mask].copy()

tracking_train = tracking_ids.loc[train_mask].copy()
tracking_validation = tracking_ids.loc[validation_mask].copy()
tracking_test = tracking_ids.loc[test_mask].copy()

# Leakage and completeness validation

assert train_patient_ids.isdisjoint(
    validation_patient_ids
), "Patient overlap found between train and validation."

assert train_patient_ids.isdisjoint(
    test_patient_ids
), "Patient overlap found between train and test."

assert validation_patient_ids.isdisjoint(
    test_patient_ids
), "Patient overlap found between validation and test."

row_assignment_count = (
    train_mask.astype(int)
    + validation_mask.astype(int)
    + test_mask.astype(int)
)

assert row_assignment_count.eq(1).all(), (
    "Some encounters were missing or assigned to multiple splits."
)

assert (
    len(X_train)
    + len(X_validation)
    + len(X_test)
    == len(df)
), "Split encounter counts do not equal the full dataset."

assert (
    len(train_patient_ids)
    + len(validation_patient_ids)
    + len(test_patient_ids)
    == groups.nunique()
), "Split patient counts do not equal the total unique patients."

# Create split summary

split_summary = pd.DataFrame({
    "split": [
        "Train",
        "Validation",
        "Test"
    ],
    "encounters": [
        len(X_train),
        len(X_validation),
        len(X_test)
    ],
    "patients": [
        groups_train.nunique(),
        groups_validation.nunique(),
        groups_test.nunique()
    ],
    "positive_cases": [
        int(y_train.sum()),
        int(y_validation.sum()),
        int(y_test.sum())
    ],
    "positive_rate_pct": [
        y_train.mean() * 100,
        y_validation.mean() * 100,
        y_test.mean() * 100
    ]
})

split_summary["encounter_percentage"] = (
    split_summary["encounters"]
    / len(df)
    * 100
)

split_summary["patient_percentage"] = (
    split_summary["patients"]
    / groups.nunique()
    * 100
)

print("\nPatient-level split completed successfully.")
print("-" * 60)

display(
    split_summary[
        [
            "split",
            "encounters",
            "encounter_percentage",
            "patients",
            "patient_percentage",
            "positive_cases",
            "positive_rate_pct"
        ]
    ].round(3)
)

print("\nPatient overlap checks:")
print(
    "Train vs. validation:",
    len(train_patient_ids & validation_patient_ids)
)
print(
    "Train vs. test:",
    len(train_patient_ids & test_patient_ids)
)
print(
    "Validation vs. test:",
    len(validation_patient_ids & test_patient_ids)
)


#Save and validate patient-level split assignments

# Define output paths

SPLIT_ASSIGNMENT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "patient_split_assignments.csv"
)

MODELING_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "modeling"
)

SPLIT_SUMMARY_PATH = (
    MODELING_REPORT_DIR
    / "split_summary.csv"
)

# Create output folder if it does not already exist
MODELING_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Create encounter-level split assignment table

split_assignments = df[
    [
        "encounter_id",
        GROUP_COLUMN,
        TARGET
    ]
].copy()

split_assignments["split"] = np.select(
    [
        split_assignments[GROUP_COLUMN].isin(
            train_patient_ids
        ),
        split_assignments[GROUP_COLUMN].isin(
            validation_patient_ids
        ),
        split_assignments[GROUP_COLUMN].isin(
            test_patient_ids
        )
    ],
    [
        "train",
        "validation",
        "test"
    ],
    default="unassigned"
)

# Validate split assignments before saving

assert len(split_assignments) == len(df), (
    "Split assignment table does not contain all encounters."
)

assert split_assignments[
    "encounter_id"
].is_unique, (
    "Duplicate encounter IDs were found."
)

assert (
    split_assignments["split"] != "unassigned"
).all(), (
    "At least one encounter was not assigned to a split."
)

assert set(
    split_assignments["split"].unique()
) == {
    "train",
    "validation",
    "test"
}, (
    "Unexpected split labels were found."
)

# Confirm each patient appears in exactly one split
patient_split_counts = (
    split_assignments.groupby(GROUP_COLUMN)["split"]
    .nunique()
)

assert patient_split_counts.max() == 1, (
    "At least one patient appears in multiple splits."
)

assert patient_split_counts.min() == 1, (
    "At least one patient has no valid split assignment."
)

# Confirm assignment counts match Step 2
assignment_counts = (
    split_assignments["split"]
    .value_counts()
)

assert assignment_counts["train"] == len(X_train)
assert assignment_counts["validation"] == len(X_validation)
assert assignment_counts["test"] == len(X_test)

# Save assignment and summary files

split_assignments.to_csv(
    SPLIT_ASSIGNMENT_PATH,
    index=False
)

split_summary.to_csv(
    SPLIT_SUMMARY_PATH,
    index=False
)

# Reload files to confirm successful saving

saved_split_assignments = pd.read_csv(
    SPLIT_ASSIGNMENT_PATH
)

saved_split_summary = pd.read_csv(
    SPLIT_SUMMARY_PATH
)

assert saved_split_assignments.shape == (
    99343,
    4
), (
    "Saved split assignment file has an unexpected shape."
)

assert saved_split_assignments[
    "encounter_id"
].is_unique

assert saved_split_assignments[
    "split"
].value_counts().to_dict() == (
    split_assignments[
        "split"
    ].value_counts().to_dict()
)

# Display saved assignment summary

saved_assignment_summary = (
    saved_split_assignments.groupby("split")
    .agg(
        encounters=("encounter_id", "size"),
        patients=(GROUP_COLUMN, "nunique"),
        positive_cases=(TARGET, "sum"),
        positive_rate=(TARGET, "mean")
    )
    .reset_index()
)

saved_assignment_summary[
    "positive_rate_pct"
] = (
    saved_assignment_summary["positive_rate"]
    * 100
)

print("Split assignments saved and validated successfully.")
print("-" * 60)

print(
    "Split assignment file:",
    SPLIT_ASSIGNMENT_PATH.resolve()
)

print(
    "Split summary file:",
    SPLIT_SUMMARY_PATH.resolve()
)

print(
    "\nSaved split-assignment shape:",
    saved_split_assignments.shape
)

print(
    "Unique encounters:",
    saved_split_assignments[
        "encounter_id"
    ].nunique()
)

print(
    "Unique patients:",
    saved_split_assignments[
        GROUP_COLUMN
    ].nunique()
)

print(
    "Maximum number of splits per patient:",
    patient_split_counts.max()
)

print("\nSaved assignment summary:")

display(
    saved_assignment_summary[
        [
            "split",
            "encounters",
            "patients",
            "positive_cases",
            "positive_rate_pct"
        ]
    ].round(3)
)

#Audit feature consistency and unseen categories across splits

# Basic split-level feature checks

split_feature_checks = pd.DataFrame({
    "split": [
        "Train",
        "Validation",
        "Test"
    ],
    "rows": [
        len(X_train),
        len(X_validation),
        len(X_test)
    ],
    "columns": [
        X_train.shape[1],
        X_validation.shape[1],
        X_test.shape[1]
    ],
    "missing_values": [
        X_train.isna().sum().sum(),
        X_validation.isna().sum().sum(),
        X_test.isna().sum().sum()
    ],
    "duplicate_rows": [
        X_train.duplicated().sum(),
        X_validation.duplicated().sum(),
        X_test.duplicated().sum()
    ]
})

print("Split-level feature checks:")
display(split_feature_checks)

# Confirm identical feature names and order

assert X_train.columns.tolist() == model_features, (
    "Training feature names or order do not match the schema."
)

assert X_validation.columns.tolist() == model_features, (
    "Validation feature names or order do not match the schema."
)

assert X_test.columns.tolist() == model_features, (
    "Test feature names or order do not match the schema."
)

assert X_train.shape[1] == 43
assert X_validation.shape[1] == 43
assert X_test.shape[1] == 43

# Confirm feature-type lists remain valid

assert set(
    numeric_model_features
).isdisjoint(
    categorical_model_features
), (
    "A feature appears in both numeric and categorical lists."
)

assert set(
    numeric_model_features
    + categorical_model_features
) == set(model_features), (
    "Numeric and categorical lists do not cover all model features."
)

assert "discharge_disposition_id" in categorical_model_features, (
    "discharge_disposition_id must be treated as categorical."
)

# Audit unseen validation and test categories
# compared with the training data

unseen_category_rows = []

for feature in categorical_model_features:

    train_categories = set(
        X_train[feature]
        .dropna()
        .astype(str)
        .unique()
    )

    validation_categories = set(
        X_validation[feature]
        .dropna()
        .astype(str)
        .unique()
    )

    test_categories = set(
        X_test[feature]
        .dropna()
        .astype(str)
        .unique()
    )

    unseen_validation = sorted(
        validation_categories - train_categories
    )

    unseen_test = sorted(
        test_categories - train_categories
    )

    unseen_category_rows.append({
        "feature": feature,
        "train_unique_categories": len(train_categories),
        "validation_unique_categories":
            len(validation_categories),
        "test_unique_categories":
            len(test_categories),
        "unseen_validation_count":
            len(unseen_validation),
        "unseen_test_count":
            len(unseen_test),
        "unseen_validation_categories":
            unseen_validation,
        "unseen_test_categories":
            unseen_test
    })

unseen_category_audit = pd.DataFrame(
    unseen_category_rows
)

print("\nCategorical feature coverage audit:")
display(
    unseen_category_audit[
        [
            "feature",
            "train_unique_categories",
            "validation_unique_categories",
            "test_unique_categories",
            "unseen_validation_count",
            "unseen_test_count"
        ]
    ]
)


# Show only features containing unseen categories
features_with_unseen_categories = (
    unseen_category_audit[
        (
            unseen_category_audit[
                "unseen_validation_count"
            ] > 0
        )
        |
        (
            unseen_category_audit[
                "unseen_test_count"
            ] > 0
        )
    ]
)

print("\nFeatures with categories not observed in training:")

if features_with_unseen_categories.empty:
    print(
        "No unseen validation or test categories were found."
    )
else:
    display(
        features_with_unseen_categories[
            [
                "feature",
                "unseen_validation_categories",
                "unseen_test_categories"
            ]
        ]
    )

# Final summary

print("\nFeature audit summary:")
print("Training features:", X_train.shape[1])
print("Validation features:", X_validation.shape[1])
print("Test features:", X_test.shape[1])

print(
    "Features with unseen validation categories:",
    int(
        (
            unseen_category_audit[
                "unseen_validation_count"
            ] > 0
        ).sum()
    )
)

print(
    "Features with unseen test categories:",
    int(
        (
            unseen_category_audit[
                "unseen_test_count"
            ] > 0
        ).sum()
    )
)

print(
    "\ndischarge_disposition_id treated as categorical:",
    "discharge_disposition_id"
    in categorical_model_features
)

print("\nStep 4 completed successfully.")

# Step 4A: Inspect duplicate predictor rows in the training partition

# Identify all rows involved in duplicated predictor patterns
duplicate_predictor_mask = X_train.duplicated(
    keep=False
)

duplicate_predictor_rows = X_train.loc[
    duplicate_predictor_mask
].copy()

print(
    "Number of training rows involved in duplicated "
    "predictor patterns:",
    len(duplicate_predictor_rows)
)

print(
    "Number of duplicated predictor patterns:",
    X_train.duplicated().sum()
)


if duplicate_predictor_rows.empty:

    print(
        "\nNo duplicated training predictor patterns found."
    )

else:

    # Add identifiers and target for investigation only
    duplicate_audit = (
        df.loc[
            duplicate_predictor_rows.index,
            [
                "encounter_id",
                "patient_nbr",
                TARGET
            ]
            + model_features
        ]
        .sort_values(model_features)
    )

    print(
        "\nIdentifiers and targets for rows sharing "
        "the same predictor values:"
    )

    display(
        duplicate_audit[
            [
                "encounter_id",
                "patient_nbr",
                TARGET
            ]
            + model_features
        ]
    )

    print("\nDuplicate-pattern audit summary:")

    print(
        "Unique encounters:",
        duplicate_audit["encounter_id"].nunique()
    )

    print(
        "Unique patients:",
        duplicate_audit["patient_nbr"].nunique()
    )

    print(
        "Target values:",
        sorted(
            duplicate_audit[TARGET].unique().tolist()
        )
    )

    print(
        "Exact duplicate complete rows:",
        duplicate_audit.duplicated().sum()
    )


# The audited full dataset must still contain no exact duplicates
assert df.duplicated().sum() == 0, (
    "Exact duplicate rows were found in the full dataset."
)

print(
    "\nFull audited dataset exact duplicates:",
    df.duplicated().sum()
)

print("\nStep 4A completed successfully.")

# Step 5: Define numeric and categorical preprocessing pipelines

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Conservative threshold for grouping extremely rare categories.
# This rule will be learned from the training data only.
RARE_CATEGORY_MIN_FREQUENCY = 10

# Numeric preprocessing

numeric_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Categorical preprocessing

categorical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=RARE_CATEGORY_MIN_FREQUENCY,
                sparse_output=True
            )
        )
    ]
)

# Combine numeric and categorical preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_preprocessor,
            numeric_model_features
        ),
        (
            "categorical",
            categorical_preprocessor,
            categorical_model_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

# Validate preprocessing configuration

assert len(numeric_model_features) == 8
assert len(categorical_model_features) == 35

assert set(
    numeric_model_features
).isdisjoint(
    categorical_model_features
)

assert set(
    numeric_model_features
    + categorical_model_features
) == set(model_features)

assert (
    "discharge_disposition_id"
    in categorical_model_features
)

assert (
    "discharge_disposition_id"
    not in numeric_model_features
)


print("Preprocessing pipeline defined successfully.")
print("-" * 60)

print(
    "Numeric features:",
    len(numeric_model_features)
)

print(
    "Categorical features:",
    len(categorical_model_features)
)

print(
    "Total raw predictors:",
    len(model_features)
)

print(
    "Rare-category minimum frequency:",
    RARE_CATEGORY_MIN_FREQUENCY
)

print("\nNumeric preprocessing:")
print("1. Median imputation")
print("2. Standard scaling")

print("\nCategorical preprocessing:")
print("1. Most-frequent imputation")
print("2. Rare-category grouping")
print("3. One-hot encoding")
print("4. Safe handling of unseen categories")

print(
    "\ndischarge_disposition_id treated as categorical:",
    "discharge_disposition_id"
    in categorical_model_features
)

print(
    "\nImportant: the preprocessor has been defined "
    "but has not yet been fitted."
)

print("\nStep 5 completed successfully.")

#Fit and validate preprocessing using training data only

from sklearn.base import clone
from scipy import sparse


# Clone the preprocessor for this validation audit

# The original preprocessor remains available for use
# inside future model pipelines.
preprocessor_audit = clone(preprocessor)

# Fit only on training data

X_train_transformed = (
    preprocessor_audit.fit_transform(X_train)
)

# Apply the fitted transformations without refitting
X_validation_transformed = (
    preprocessor_audit.transform(X_validation)
)

X_test_transformed = (
    preprocessor_audit.transform(X_test)
)

# Retrieve transformed feature names

transformed_feature_names = (
    preprocessor_audit.get_feature_names_out()
)

transformed_feature_count = len(
    transformed_feature_names
)

# Validate transformed matrix dimensions

assert X_train_transformed.shape[0] == len(X_train)
assert X_validation_transformed.shape[0] == len(X_validation)
assert X_test_transformed.shape[0] == len(X_test)

assert (
    X_train_transformed.shape[1]
    == X_validation_transformed.shape[1]
    == X_test_transformed.shape[1]
), (
    "Transformed feature counts differ across splits."
)

assert (
    X_train_transformed.shape[1]
    == transformed_feature_count
), (
    "Feature-name count does not match transformed matrix width."
)

assert len(set(transformed_feature_names)) == (
    transformed_feature_count
), (
    "Duplicate transformed feature names were found."
)

# Check for invalid transformed values

def contains_non_finite_values(matrix):
    """
    Return True if a dense or sparse matrix contains
    NaN, positive infinity, or negative infinity.
    """

    if sparse.issparse(matrix):
        return not np.isfinite(matrix.data).all()

    return not np.isfinite(matrix).all()


assert not contains_non_finite_values(
    X_train_transformed
), "Invalid values found in transformed training data."

assert not contains_non_finite_values(
    X_validation_transformed
), "Invalid values found in transformed validation data."

assert not contains_non_finite_values(
    X_test_transformed
), "Invalid values found in transformed test data."

# Calculate transformed matrix density

def matrix_density(matrix):
    """
    Calculate the proportion of non-zero values.
    """

    total_values = matrix.shape[0] * matrix.shape[1]

    if sparse.issparse(matrix):
        return matrix.nnz / total_values

    return np.count_nonzero(matrix) / total_values


transformed_summary = pd.DataFrame({
    "split": [
        "Train",
        "Validation",
        "Test"
    ],
    "rows": [
        X_train_transformed.shape[0],
        X_validation_transformed.shape[0],
        X_test_transformed.shape[0]
    ],
    "transformed_features": [
        X_train_transformed.shape[1],
        X_validation_transformed.shape[1],
        X_test_transformed.shape[1]
    ],
    "sparse_matrix": [
        sparse.issparse(X_train_transformed),
        sparse.issparse(X_validation_transformed),
        sparse.issparse(X_test_transformed)
    ],
    "density_pct": [
        matrix_density(X_train_transformed) * 100,
        matrix_density(X_validation_transformed) * 100,
        matrix_density(X_test_transformed) * 100
    ],
    "contains_non_finite_values": [
        contains_non_finite_values(
            X_train_transformed
        ),
        contains_non_finite_values(
            X_validation_transformed
        ),
        contains_non_finite_values(
            X_test_transformed
        )
    ]
})

# Audit infrequent-category grouping

fitted_encoder = (
    preprocessor_audit
    .named_transformers_["categorical"]
    .named_steps["encoder"]
)

infrequent_category_information = getattr(
    fitted_encoder,
    "infrequent_categories_",
    [None] * len(categorical_model_features)
)

rare_category_rows = []

for feature, categories, infrequent_categories in zip(
    categorical_model_features,
    fitted_encoder.categories_,
    infrequent_category_information
):

    infrequent_count = (
        0
        if infrequent_categories is None
        else len(infrequent_categories)
    )

    rare_category_rows.append({
        "feature": feature,
        "training_categories": len(categories),
        "infrequent_categories_grouped":
            infrequent_count
    })

rare_category_summary = pd.DataFrame(
    rare_category_rows
)

features_with_grouped_categories = (
    rare_category_summary[
        rare_category_summary[
            "infrequent_categories_grouped"
        ] > 0
    ]
    .sort_values(
        "infrequent_categories_grouped",
        ascending=False
    )
    .reset_index(drop=True)
)

# Display validation results
print(
    "Preprocessor fitted on training data only "
    "and validated successfully."
)

print("-" * 65)

display(
    transformed_summary.round(4)
)

print(
    "\nRaw predictor count:",
    len(model_features)
)

print(
    "Transformed predictor count:",
    transformed_feature_count
)

print(
    "Numeric transformed features:",
    len(numeric_model_features)
)

print(
    "One-hot encoded categorical features:",
    transformed_feature_count
    - len(numeric_model_features)
)

print(
    "\nFeatures containing grouped infrequent categories:",
    len(features_with_grouped_categories)
)

if features_with_grouped_categories.empty:
    print(
        "No training categories were below the "
        "minimum-frequency threshold."
    )
else:
    display(features_with_grouped_categories)


print("\nFirst 20 transformed feature names:")

for feature_name in transformed_feature_names[:20]:
    print(feature_name)


print(
    "\nImportant: validation and test data were "
    "transformed but were not used to fit preprocessing."
)

print("\nStep 6 completed successfully.")

#Save preprocessing metadata and transformed feature information

import json
from pathlib import Path

# Define output folders and paths

ARTIFACT_DIR = (
    PROJECT_ROOT
    / "artifacts"
)

MODELING_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "modeling"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELING_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PREPROCESSING_METADATA_PATH = (
    ARTIFACT_DIR
    / "preprocessing_metadata.json"
)

TRANSFORMED_FEATURES_PATH = (
    MODELING_REPORT_DIR
    / "transformed_feature_names.csv"
)

RARE_CATEGORY_SUMMARY_PATH = (
    MODELING_REPORT_DIR
    / "rare_category_summary.csv"
)

PREPROCESSING_VALIDATION_PATH = (
    MODELING_REPORT_DIR
    / "preprocessing_validation_summary.csv"
)

# Create preprocessing metadata

preprocessing_metadata = {
    "random_state": RANDOM_STATE,
    "target_column": TARGET,
    "group_column": GROUP_COLUMN,

    "raw_feature_count": len(model_features),
    "numeric_feature_count":
        len(numeric_model_features),
    "categorical_feature_count":
        len(categorical_model_features),
    "transformed_feature_count":
        transformed_feature_count,

    "numeric_features":
        numeric_model_features,
    "categorical_features":
        categorical_model_features,

    "numeric_preprocessing": {
        "imputation_strategy": "median",
        "scaling": "StandardScaler"
    },

    "categorical_preprocessing": {
        "imputation_strategy": "most_frequent",
        "encoding": "OneHotEncoder",
        "handle_unknown": "infrequent_if_exist",
        "minimum_frequency":
            RARE_CATEGORY_MIN_FREQUENCY,
        "sparse_output": True
    },

    "split_sizes": {
        "train_encounters": len(X_train),
        "validation_encounters":
            len(X_validation),
        "test_encounters": len(X_test),

        "train_patients":
            int(groups_train.nunique()),
        "validation_patients":
            int(groups_validation.nunique()),
        "test_patients":
            int(groups_test.nunique())
    },

    "target_rates": {
        "train_positive_rate":
            float(y_train.mean()),
        "validation_positive_rate":
            float(y_validation.mean()),
        "test_positive_rate":
            float(y_test.mean())
    },

    "features_with_grouped_categories":
        int(len(features_with_grouped_categories)),

    "prediction_timing":
        "At hospital discharge",

    "important_notes": [
        (
            "patient_nbr is used only for "
            "patient-level splitting"
        ),
        (
            "encounter_id is used only for "
            "record tracking"
        ),
        (
            "discharge_disposition_id is treated "
            "as categorical"
        ),
        (
            "Preprocessing must be fitted using "
            "training data only"
        ),
        (
            "Validation and test data must not "
            "be used to fit preprocessing"
        ),
        (
            "Class resampling must be applied "
            "only within training workflows"
        )
    ]
}

# Save transformed feature names

transformed_feature_table = pd.DataFrame({
    "feature_index": range(
        transformed_feature_count
    ),
    "transformed_feature_name":
        transformed_feature_names
})

# Save files

with open(
    PREPROCESSING_METADATA_PATH,
    "w",
    encoding="utf-8"
) as metadata_file:

    json.dump(
        preprocessing_metadata,
        metadata_file,
        indent=4
    )


transformed_feature_table.to_csv(
    TRANSFORMED_FEATURES_PATH,
    index=False
)

rare_category_summary.to_csv(
    RARE_CATEGORY_SUMMARY_PATH,
    index=False
)

transformed_summary.to_csv(
    PREPROCESSING_VALIDATION_PATH,
    index=False
)

# Reload and validate saved outputs

with open(
    PREPROCESSING_METADATA_PATH,
    "r",
    encoding="utf-8"
) as metadata_file:

    saved_preprocessing_metadata = json.load(
        metadata_file
    )


saved_transformed_features = pd.read_csv(
    TRANSFORMED_FEATURES_PATH
)

saved_rare_category_summary = pd.read_csv(
    RARE_CATEGORY_SUMMARY_PATH
)

saved_preprocessing_validation = pd.read_csv(
    PREPROCESSING_VALIDATION_PATH
)


assert (
    saved_preprocessing_metadata[
        "raw_feature_count"
    ] == 43
)

assert (
    saved_preprocessing_metadata[
        "transformed_feature_count"
    ] == 179
)

assert (
    len(saved_transformed_features)
    == transformed_feature_count
)

assert (
    saved_transformed_features[
        "transformed_feature_name"
    ].is_unique
)

assert (
    len(saved_rare_category_summary)
    == len(categorical_model_features)
)

assert (
    len(saved_preprocessing_validation)
    == 3
)

# Display saved-output summary

print(
    "Preprocessing metadata and reports "
    "saved successfully."
)

print("-" * 65)

print(
    "Metadata file:",
    PREPROCESSING_METADATA_PATH.resolve()
)

print(
    "Transformed feature names:",
    TRANSFORMED_FEATURES_PATH.resolve()
)

print(
    "Rare-category summary:",
    RARE_CATEGORY_SUMMARY_PATH.resolve()
)

print(
    "Preprocessing validation summary:",
    PREPROCESSING_VALIDATION_PATH.resolve()
)

print(
    "\nRaw predictors:",
    saved_preprocessing_metadata[
        "raw_feature_count"
    ]
)

print(
    "Transformed predictors:",
    saved_preprocessing_metadata[
        "transformed_feature_count"
    ]
)

print(
    "Features with grouped rare categories:",
    saved_preprocessing_metadata[
        "features_with_grouped_categories"
    ]
)

print(
    "\nSaved transformed feature rows:",
    len(saved_transformed_features)
)

print(
    "Saved rare-category summary rows:",
    len(saved_rare_category_summary)
)

print("\nStep 7 completed successfully.")

#Final validation audit

# Verify split totals

total_split_rows = (
    len(X_train)
    + len(X_validation)
    + len(X_test)
)

total_split_patients = (
    groups_train.nunique()
    + groups_validation.nunique()
    + groups_test.nunique()
)

assert total_split_rows == len(df), (
    "Train, validation, and test rows do not equal the full dataset."
)

assert total_split_patients == groups.nunique(), (
    "Train, validation, and test patient counts do not equal "
    "the total number of patients."
)

# Verify patient separation

assert set(groups_train).isdisjoint(
    set(groups_validation)
)

assert set(groups_train).isdisjoint(
    set(groups_test)
)

assert set(groups_validation).isdisjoint(
    set(groups_test)
)

# Verify raw feature structure

assert X_train.shape[1] == 43
assert X_validation.shape[1] == 43
assert X_test.shape[1] == 43

assert len(numeric_model_features) == 8
assert len(categorical_model_features) == 35

assert TARGET not in X_train.columns
assert GROUP_COLUMN not in X_train.columns
assert "encounter_id" not in X_train.columns

# Verify transformed feature structure

assert X_train_transformed.shape == (
    len(X_train),
    179
)

assert X_validation_transformed.shape == (
    len(X_validation),
    179
)

assert X_test_transformed.shape == (
    len(X_test),
    179
)

assert len(transformed_feature_names) == 179
assert len(set(transformed_feature_names)) == 179

# Verify saved files

required_saved_files = [
    SPLIT_ASSIGNMENT_PATH,
    SPLIT_SUMMARY_PATH,
    PREPROCESSING_METADATA_PATH,
    TRANSFORMED_FEATURES_PATH,
    RARE_CATEGORY_SUMMARY_PATH,
    PREPROCESSING_VALIDATION_PATH
]

missing_saved_files = [
    path
    for path in required_saved_files
    if not path.exists()
]

assert not missing_saved_files, (
    f"Missing saved files: {missing_saved_files}"
)

# Create final Notebook 3 completion summary

notebook_3_summary = pd.DataFrame({
    "check": [
        "Full dataset rows",
        "Unique patients",
        "Raw predictors",
        "Numeric predictors",
        "Categorical predictors",
        "Transformed predictors",
        "Train encounters",
        "Validation encounters",
        "Test encounters",
        "Train patients",
        "Validation patients",
        "Test patients",
        "Train positive rate",
        "Validation positive rate",
        "Test positive rate",
        "Patient overlap",
        "Missing raw values",
        "Non-finite transformed values",
        "Saved output files"
    ],
    "result": [
        len(df),
        groups.nunique(),
        len(model_features),
        len(numeric_model_features),
        len(categorical_model_features),
        len(transformed_feature_names),
        len(X_train),
        len(X_validation),
        len(X_test),
        groups_train.nunique(),
        groups_validation.nunique(),
        groups_test.nunique(),
        f"{y_train.mean():.4%}",
        f"{y_validation.mean():.4%}",
        f"{y_test.mean():.4%}",
        0,
        int(df.isna().sum().sum()),
        0,
        len(required_saved_files)
    ],
    "status": [
        "Passed"
    ] * 19
})

NOTEBOOK_3_SUMMARY_PATH = (
    PROJECT_ROOT
    / "reports"
    / "modeling"
    / "notebook_3_completion_summary.csv"
)

notebook_3_summary.to_csv(
    NOTEBOOK_3_SUMMARY_PATH,
    index=False
)

# Display final results

print("Notebook 3 final validation completed successfully.")
print("-" * 65)

display(notebook_3_summary)

print(
    "\nCompletion summary saved to:",
    NOTEBOOK_3_SUMMARY_PATH.resolve()
)

print("\nImportant modeling rules carried forward:")
print("1. Use the saved patient-level split assignments.")
print("2. Do not regenerate the train, validation, and test sets.")
print("3. Fit preprocessing only within the training workflow.")
print("4. Keep the test set untouched until final evaluation.")
print("5. Apply class weighting or resampling only to training data.")
print("6. Evaluate imbalance-aware metrics, not accuracy alone.")












Current working directory: c:\Users\pradh\Documents\hospital-readmission-project\notebooks
Project root: C:\Users\pradh\Documents\hospital-readmission-project
Dataset exists: True
Feature schema exists: True

Modeling inputs loaded successfully.
------------------------------------------------------------
Pandas version: 3.0.3
Scikit-learn version: 1.9.0
Random state: 42

Dataset shape: (99343, 46)
Predictor matrix shape: (99343, 43)
Target shape: (99343,)
Patient-group shape: (99343,)
Tracking-ID shape: (99343,)

Unique patients: 69990
Numeric predictors: 8
Categorical predictors: 35
Total predictors: 43

Target distribution:
readmitted_30
Not readmitted    88029
Readmitted        11314
Name: count, dtype: int64

Positive-class rate: 11.3888%

completed successfully.
Patient-level split table created.
Unique patients: 69990

Patient split strata:
split_stratum
0_1_encounter             51362
0_2_encounters             7284
0_3_or_more_encounters     2539
1_1_encounter              228

,split,encounters,encounter_percentage,patients,patient_percentage,positive_cases,positive_rate_pct
0,Train,69467,69.9260,48993,70.0000,7936,11.4240
1,Validation,14900,14.9990,10498,14.9990,1680,11.2750
2,Test,14976,15.0750,10499,15.0010,1698,11.3380



Patient overlap checks:
Train vs. validation: 0
Train vs. test: 0
Validation vs. test: 0
Split assignments saved and validated successfully.
------------------------------------------------------------
Split assignment file: C:\Users\pradh\Documents\hospital-readmission-project\data\processed\patient_split_assignments.csv
Split summary file: C:\Users\pradh\Documents\hospital-readmission-project\reports\modeling\split_summary.csv

Saved split-assignment shape: (99343, 4)
Unique encounters: 99343
Unique patients: 69990
Maximum number of splits per patient: 1

Saved assignment summary:


,split,encounters,patients,positive_cases,positive_rate_pct
0,test,14976,10499,1698,11.3380
1,train,69467,48993,7936,11.4240
2,validation,14900,10498,1680,11.2750


Split-level feature checks:


...


Raw predictor count: 43
Transformed predictor count: 179
Numeric transformed features: 8
One-hot encoded categorical features: 171

Features containing grouped infrequent categories: 14


,feature,training_categories,infrequent_categories_grouped
0,discharge_disposition_id,21,5
1,chlorpropamide,4,2
2,acarbose,4,2
3,miglitol,4,2
4,glyburide-metformin,4,2
5,gender,3,1
6,acetohexamide,2,1
7,nateglinide,4,1
8,troglitazone,2,1
9,tolazamide,3,1



First 20 transformed feature names:
numeric__time_in_hospital
numeric__num_lab_procedures
numeric__num_procedures
numeric__num_medications
numeric__number_outpatient
numeric__number_emergency
numeric__number_inpatient
numeric__number_diagnoses
categorical__race_AfricanAmerican
categorical__race_Asian
categorical__race_Caucasian
categorical__race_Hispanic
categorical__race_Other
categorical__race_Unknown
categorical__gender_Female
categorical__gender_Male
categorical__gender_infrequent_sklearn
categorical__age_[0-10)
categorical__age_[10-20)
categorical__age_[20-30)

Important: validation and test data were transformed but were not used to fit preprocessing.

Step 6 completed successfully.
Preprocessing metadata and reports saved successfully.
-----------------------------------------------------------------
Metadata file: C:\Users\pradh\Documents\hospital-readmission-project\artifacts\preprocessing_metadata.json
Transformed feature names: C:\Users\pradh\Documents\hospital-readmission-

,check,result,status
0,Full dataset rows,99343,Passed
1,Unique patients,69990,Passed
2,Raw predictors,43,Passed
3,Numeric predictors,8,Passed
4,Categorical predictors,35,Passed
5,Transformed predictors,179,Passed
6,Train encounters,69467,Passed
7,Validation encounters,14900,Passed
8,Test encounters,14976,Passed
9,Train patients,48993,Passed



Completion summary saved to: C:\Users\pradh\Documents\hospital-readmission-project\reports\modeling\notebook_3_completion_summary.csv

Important modeling rules carried forward:
1. Use the saved patient-level split assignments.
2. Do not regenerate the train, validation, and test sets.
3. Fit preprocessing only within the training workflow.
4. Keep the test set untouched until final evaluation.
5. Apply class weighting or resampling only to training data.
6. Evaluate imbalance-aware metrics, not accuracy alone.
